# **Model**

# **XRF milk database**

In [5]:
# importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks

# loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv('XRF_databases/milk/plsda/milk.csv', sep=';') # local copy of Toledo 2022 dataset
data = data_complete.loc[:, '1.00':'24.00']

# Creating a new column 'Class' based on the condition of the samples in the 'Type' column being 'Authentic'
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1.00':'24.00'], test_size=0.30) # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1.00':'24.00'], test_size=0.30) # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True) # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0]) # creating the target variable for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0]) # creating the target variable for prediction set

# preprocessings
import preprocessings as prepr # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

from modeling import pls_optimized

# performing PLS-DA with optimized latent variables
plsda_results = pls_optimized(Xcalclass_prep, 
                              ycalclass,
                              LVmax=4,
                              Xpred=Xpredclass_prep,
                              ypred=ypredclass,
                              aim='classification',
                              cv=10)
plsda_results[0]

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-05 18:16:23,706 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-05 18:16:23,736 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

,LVs,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy CV,Sensitivity CV,Specificity CV,CM CV,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred,X Cum Exp Var,Y Cum Exp Var,X Ind Exp Var,Y Ind Exp Var
0,1,0.671642,0.773810,0.50,"[[50, 50], [38, 130]]",0.507463,0.684524,0.21,"[[21, 79], [53, 115]]",0.617391,0.722222,0.441860,"[[19, 24], [20, 52]]",10.552826,8.090290,10.552826,8.090290
1,2,0.798507,0.880952,0.66,"[[66, 34], [20, 148]]",0.597015,0.738095,0.36,"[[36, 64], [44, 124]]",0.678261,0.819444,0.441860,"[[19, 24], [13, 59]]",17.967265,11.982751,7.414439,3.892461
2,3,0.824627,0.869048,0.75,"[[75, 25], [22, 146]]",0.649254,0.738095,0.50,"[[50, 50], [44, 124]]",0.678261,0.833333,0.418605,"[[18, 25], [12, 60]]",27.994666,16.772999,10.027401,4.790248
3,4,0.944030,0.946429,0.94,"[[94, 6], [9, 159]]",0.690299,0.791667,0.52,"[[52, 48], [35, 133]]",0.773913,0.875000,0.604651,"[[26, 17], [9, 63]]",32.802808,25.137616,4.808143,8.364617


# **VIP and SHAP**

In [7]:
pd.options.plotting.backend = 'plotly' # setting plotly as the backend for pandas plotting 
Xcalclass.T.plot()

In [8]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('background1', 1.0, 2.66),
('Ag La', 2.68, 3.10),
('Ag Lb', 3.12, 3.46),
('Ca', 3.48, 3.92),
('background2', 3.94, 6.12),
('Fe ka', 6.14, 6.68),
('background3', 6.70, 20.06),
('Ag compton', 20.08, 21.62),
('Ag ka', 21.48, 22.62),
('background5', 22.64, 24.00),
]

In [9]:
import numpy as np
import pandas as pd

# vip
vip_scores_df = pd.DataFrame({
    'energy' : plsda_results[4].T.index,
    'VIP_Score' : plsda_results[4].T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# vamos gerar uma nova coluna em vip_scores_df com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_vip = {} # dicionário para mapear energia para zona espectral
for zone_name, start, end in spectral_cuts: # iterando sobre cada zona espectral (que tem nome, início e fim)
	for i in vip_scores_df['energy']: # iterando sobre cada valor de energia no vip_scores_df
		i_float = float(i)
		if start <= i_float <= end:
			energy_to_zone_vip[i] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip) # a funcao map funciona como um buscador que substitui os valores de 'energy' pelos valores correspondentes no dicionário energy_to_zone_vip

# vamos filtrar vip_scores_df para manter apenas as zonas espectrais únicas com maior VIP score
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# reg vet
reg_vet = pd.DataFrame(plsda_results[3].coef_, columns=plsda_results[3].feature_names_in_) # creating a DataFrame with regression coefficients
reg_vet = reg_vet.T
reg_vet.insert(0, 'energy', reg_vet.index) # adding energy column
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy', 'Reg_coef'] # renaming
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs() # adding absolute value column
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True) # sorting by absolute value

# gerando uma nova coluna em reg_vet com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_reg = {} # dicionário para mapear energia para zona espectral
for zone_name, start, end in spectral_cuts: # iterando sobre cada zona espectral (que tem nome, início e fim)
    for i in reg_vet['energy']: # iterando sobre cada valor de energia no reg_vet
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_reg[i] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg) # a funcao map funciona como um buscador que substitui os valores de 'energy' pelos valores correspondentes no dicionário energy_to_zone_reg
reg_vet

# vamos filtrar reg_vet para manter apenas as zonas espectrais únicas com maior valor absoluto do coeficiente de regressão
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)  

In [ ]:
# vamos agora extrair as variaveis mais importantes atraves do método SHAP
import shap

# Para PLSRegression, usamos KernelExplainer porque não há explainer dedicado muito rápido
explainer_pls = shap.KernelExplainer(plsda_results[3].predict, Xcalclass_prep)
shap_values_pls = explainer_pls(Xcalclass_prep)

shap_global_importance = pd.DataFrame({
    'energy': Xpredclass_prep.columns,
    'Mean_Abs_SHAP': np.abs(shap_values_pls.values).mean(axis=0)}) # tomando a importancia global como a media dos valores absolutos dos valores SHAP para cada feature
shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False, inplace=True)

# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)

# 15 MIN  

Using 284 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/284 [00:00<?, ?it/s]

In [11]:
vip_scores_unique_df

,energy,VIP_Score,Zone
0,2.99,5.310402,Ag La
1,22.36,4.279605,Ag ka
2,3.17,4.109411,Ag Lb
3,8.75,1.763300,background3
4,6.35,1.731416,Fe ka
5,21.31,1.726519,Ag compton
6,2.46,1.532588,background1
7,22.82,1.135355,background5
8,3.89,1.000545,Ca
9,4.81,0.910534,background2


In [ ]:
vip_scores_unique_df.to_csv('XRF_databases/bank_notes/plsda/vip_scores_bank_notes.csv', index=False, sep=';')
reg_vet_unique_df.to_csv('XRF_databases/bank_notes/plsda/reg_vet_bank_notes.csv', index=False, sep=';')
shap_unique_df.to_csv('XRF_databases/bank_notes/plsda/shap_bank_notes.csv', index=False, sep=';')

In [ ]:
#shap_unique_df = pd.read_csv('XRF_databases/bank_notes/plsda/shap_bank_notes.csv', sep=';') # loading previously saved shap_unique_df
#vip_scores_unique_df = pd.read_csv('XRF_databases/bank_notes/plsda/vip_scores_bank_notes.csv', sep=';') # loading previously saved vip_scores_unique_df#
#reg_vet_unique_df = pd.read_csv('XRF_databases/bank_notes/plsda/reg_vet_bank_notes.csv', sep=';') # loading previously saved reg_vet_unique_df

# **SMeX**

In [ ]:
from explaining import extract_spectral_zones
from explaining import aggregate_spectral_zones
from explaining import predicates_by_quantiles
from explaining import create_predicate_info_dict
from explaining import bagging_predicates, calculate_predicate_metrics
from explaining import build_predicate_graph
import numpy as np
import pandas as pd

spectral_zones_class = extract_spectral_zones(Xcalclass_prep, spectral_cuts) # extracting the spectral zones
zone_sums_df = aggregate_spectral_zones(spectral_zones_class, aggregator='max')
predicates_quantiles = predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8]) # getting predicates for quartiles
co_occurrence_matrix_df=predicates_quantiles[2]

# Criar dicionário de informações
predicate_info_dict = create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=plsda_results[5].iloc[:, -1]
)

# LISTA DE SEMENTES A TESTAR

random_seeds = [0, 1, 42]

all_results = {}

training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    
    # Bagging
    bags_result_seed = bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=60,
        #n_predicates_per_bag=int(training_samples*0.6), # 60 % da base para predicados (convertido para int)
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.25), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    mi_results_dict_seed = calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance',
        threshold=0.001, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results[seed] = {
        'bags_result': bags_result_seed,
        'mi_results_dict': mi_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)

# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente
    DG = build_predicate_graph(
        bags_result=all_results[seed]['bags_result'],
        mi_results_dict=all_results[seed]['mi_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    
    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# vamos calcular a LRC de acordo com as diferentes sementes
import networkx as nx

lrc_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    local_reaching_centrality = {
        node: nx.local_reaching_centrality(DG, node, weight='weight') 
        for node in DG.nodes()
    }

    # Ordenar por LRC
    sorted_lrc = sorted(local_reaching_centrality.items(), key=lambda x: x[1], reverse=True)
    
    # Criar DataFrame com LRC
    lrc_df_seed = pd.DataFrame(sorted_lrc, columns=['Node', 'Local_Reaching_Centrality'])
    
    # Extrair informações dos predicados (zona, threshold, operador)
    zones = []
    thresholds = []
    operators = []
    
    for node in lrc_df_seed['Node']:
        if node.startswith('Class_'):
            zones.append(None)
            thresholds.append(None)
            operators.append(None)
        else:
            pred_row = predicates_quantiles[0][predicates_quantiles[0]['rule'] == node].iloc[0]
            zones.append(pred_row['zone'])
            thresholds.append(pred_row['thresholds'])
            operators.append(pred_row['operator'])
    
    lrc_df_seed['Zone'] = zones
    lrc_df_seed['Threshold'] = thresholds
    lrc_df_seed['Operator'] = operators
    lrc_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    
    # Armazenar LRC
    lrc_by_seed[seed] = lrc_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe

lrc_all_seeds_df = pd.DataFrame() # 
for seed in random_seeds:
    lrc_df_seed = lrc_by_seed[seed]
    lrc_df_seed = lrc_df_seed.rename(columns={
        'Node': f'Predicate_Seed_{seed}'
    })
    lrc_all_seeds_df = pd.concat([lrc_all_seeds_df, lrc_df_seed[[f'Predicate_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_unique_by_seed = {}
for seed, lrc_df in lrc_by_seed.items():
    lrc_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_unique_df = lrc_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_unique_by_seed[seed] = lrc_unique_df

lrc_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes   


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_11 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_12 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 60 | Descartados: 20
Bag_13 | Amo

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_Seed_0,Predicate_Seed_1,Predicate_Seed_42
0,Ag ka > 3.53,Ag ka > 3.53,Ag ka > 3.53
1,Ag ka > 2.38,Ag ka > 2.38,Ag ka > 2.38
2,Ag Lb <= 3.66,Ag Lb <= 3.66,Ag Lb <= 3.66
3,Ag La <= 2.70,Ag La <= 2.70,Ag La <= 5.24
4,Ag Lb <= 2.74,Ag Lb <= 2.74,Ag La <= 2.70
...,...,...,...
58,background5 > 2.21,background2 <= 2.15,background2 <= 2.36
59,Ag La > 2.70,Ag La > 4.22,Ag La > 2.70
60,Fe ka <= 1.19,Class_A,Class_A
61,Class_A,Class_B,Class_B


In [ ]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].values,
    'Shap' : shap_unique_df['Zone'].values
    })

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    zones = lrc_unique_df['Zone'].values
    # condição para padronizar o tamanho das listas (usando o tamanho do features_importance)
    if len(zones) < len(features_importance):
        zones = np.concatenate([zones, [None]*(len(features_importance)-len(zones))]) # adicionando None para completar o tamanho
    features_importance[f'LRC_Seed_{seed}'] = zones[:len(features_importance)]
features_importance

,Vip,Reg_coef,LRC_Seed_0,LRC_Seed_1,LRC_Seed_42
0,Fe ka,Ti ka,Fe ka,Fe ka,Fe ka
1,Ca ka,Ti kb,Ca ka,Ca ka,Ca ka
2,Ti ka,Ag ka scattering,Fe kb,Fe kb,Fe kb
3,Cu,Ca ka,Ca kb,Ca kb,Ca kb
4,Fe kb,Cu,Ti kb,Cu,Cu
5,Ca kb,Ca kb,Cu,Ti kb,Ti kb
6,Ti kb,background4,Ti ka,Ti ka,Ti ka
7,Ag ka scattering,background2,Ag ka scattering,Ag ka scattering,background4
8,background4,Ar ka + Ag L,background4,background3,Ag ka scattering
9,background2,background5,background2,background4,background3


## **RBO**

In [ ]:
# utilizando o Rank-Biased Overlap (RBO) para comparar as listas de importância de características tendo o vip como referencia
# com p = 1 é rbo equivalente ao overlap simples (interseção sobre união) sem peso para posições iniciais
# quanto menor o p, mais peso é dado para as posições iniciais da lista (mais relevante para nosso caso)
import rbo

rbo_results = {}
reference_list = [x for x in features_importance['Vip'].tolist() if pd.notnull(x)]
methods = ['Reg_coef', 'Shap'] + [f'LRC_Seed_{seed}' for seed in random_seeds]
rbo_results = {}
for method in methods:
    compare_list = [x for x in features_importance[method].tolist() if pd.notnull(x)]
    # Ensure both lists are the same length for RBO calculation
    min_len = min(len(reference_list), len(compare_list))
    ref_trimmed = reference_list[:min_len]
    comp_trimmed = compare_list[:min_len]
    score = rbo.RankingSimilarity(ref_trimmed, comp_trimmed).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method', 'RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results

,Reference,Method,RBO_Score
2,Vip,LRC_Seed_1,0.944540
3,Vip,LRC_Seed_42,0.944461
1,Vip,LRC_Seed_0,0.941527
0,Vip,Reg_coef,0.067436


In [ ]:
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from both lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Ensure both lists are the same length for RBO calculation
        min_len = min(len(list_1), len(list_2))
        list_1_trimmed = list_1[:min_len]
        list_2_trimmed = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trimmed, list_2_trimmed).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_4828\646247094.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({


,Method_1,Method_2,RBO_Score
7,LRC_Seed_0,LRC_Seed_1,0.779999
8,LRC_Seed_0,LRC_Seed_42,0.665242
9,LRC_Seed_1,LRC_Seed_42,0.603655
1,Vip,LRC_Seed_0,0.395146
4,Reg_coef,LRC_Seed_0,0.394111
3,Vip,LRC_Seed_42,0.382414
2,Vip,LRC_Seed_1,0.378685
6,Reg_coef,LRC_Seed_42,0.375699
0,Vip,Reg_coef,0.303188
5,Reg_coef,LRC_Seed_1,0.290456
